# Colab runner (thin)
Runtime -> Change runtime type -> **A100** (or T4). Secrets (key icon): `HF_TOKEN`, `GH_TOKEN` if private.
Outputs go to Google Drive (`MyDrive/mcp-poison-detect/outputs`) so they survive disconnects.

In [ ]:
REPO = "github.com/Arn15/mcp-poison-detect"
BRANCH = "main"
SCRIPT = "scripts/check_env.py"
ARGS = "--config configs/default.yaml"
GPU_DEPS = False

In [ ]:
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
def secret(k):
    try: return userdata.get(k)
    except Exception: return None
os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
gh = secret("GH_TOKEN")
url = f"https://{gh}@{REPO}.git" if gh else f"https://{REPO}.git"
os.chdir("/content")
if os.path.isdir("mcp-poison-detect/.git"):
    subprocess.run(f"git -C mcp-poison-detect pull -q", shell=True, check=True)
else:
    subprocess.run(f"git clone -q -b {BRANCH} {url} mcp-poison-detect", shell=True, check=True)
os.chdir("/content/mcp-poison-detect")
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements.txt
if GPU_DEPS:
    !pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python {SCRIPT} {ARGS}